# Day 3 · Afternoon exercises (Exercise B)

**13:30–15:00 · then everyone back in the room at 15:00 for Peer review B**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Today you run Stan yourself, with `cmdstanr`. Keep the folder `models/` next to this notebook.

**If Stan does not compile on your laptop**, do not lose the session on it: continue on UCloud (see the course page) or work with a group member's laptop, and ask us about your installation during the break.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| B1 | Run the Stan model and compare with the exact answer | 20 min |
| B2 | Check the diagnostics before interpreting | 15 min |
| B3 | Change the prior | 15 min |
| B4 | **Challenge, no template:** your own Stan program for two rates | 25 min |
| — | Exit response and your group's question set | 10 min |

**Roles this block:** rotate them from the morning.

**The example** is the same as this morning: y = 7 successes in n = 10 trials, prior Beta(1, 1), exact posterior **Beta(8, 4)**.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
library(cmdstanr)    # runs Stan from R

## B1 (20 min): run the Stan model and compare with the exact answer

**The model file** `models/beta-binomial.stan` (*slides "A Stan program has blocks" and "Running it from R"*):

```stan
data {
  int<lower=0> n;                 // number of trials
  int<lower=0, upper=n> y;        // number of successes
  real<lower=0> a;                // prior: Beta(a, b)
  real<lower=0> b;
}
parameters {
  real<lower=0, upper=1> p;       // success probability
}
model {
  p ~ beta(a, b);                 // prior
  y ~ binomial(n, p);             // likelihood
}
generated quantities {
  int y_new = binomial_rng(n, p); // successes in n new trials
}
```

**Tasks.**

1. For each block, write in one line what it contains for *this* problem (which numbers are data, what is the parameter, which line is the prior, which the likelihood, what is generated).
2. Compile the model and run it with n = 10, y = 7, a = 1, b = 1: four chains, `seed = 2026`. The first compilation takes about a minute.
3. From the draws, compute the posterior mean of p and its 90% interval. Compare with the exact Beta(8, 4): mean `8 / 12`, interval `qbeta(c(0.05, 0.95), 8, 4)`.

**Your answer:** write one line per block here.

In [ ]:
mod <- cmdstan_model("models/beta-binomial.stan")     # compile (once)

fit <- mod$sample(
  data   = list(n = 10, y = 7, a = 1, b = 1),
  chains = 4, seed = 2026, refresh = 0
)

draws <- fit$draws(format = "df")    # one row per draw: columns p, y_new, .chain, ...

# Task 3:
# mean(draws$p);  quantile(draws$p, c(0.05, 0.95))
# 8 / 12;         qbeta(c(0.05, 0.95), 8, 4)

**Your answer:** write a few sentences here.

## B2 (15 min): check the diagnostics before interpreting

**The six numbers** (*slides "Six numbers to check after every fit", "R-hat: do the chains agree?" and "Reading a real Stan summary"*):

| diagnostic | look for | where |
|---|---|---|
| R-hat | < 1.01 | `fit$summary()` |
| bulk ESS, tail ESS | > 400 | `fit$summary()` |
| MCSE of the mean | small compared with the posterior SD | `fit$summary()` |
| divergences | 0 in every chain | `fit$diagnostic_summary()` |
| max treedepth hits | 0 in every chain | `fit$diagnostic_summary()` |
| E-BFMI | > 0.3 in every chain | `fit$diagnostic_summary()` |

**Tasks.**

1. Make a trace plot of p with one colour per chain. Do the four chains look alike?
2. Fill in the table below with the values from your fit.
3. Can the results of B1 be trusted? Answer in one or two sentences, based on the numbers.
4. Name one question about the model that these diagnostics **cannot** answer (*slide "Two different checks"*).

*Optional:* run the model again with `iter_warmup = 20, iter_sampling = 50` added to `mod$sample()`. Which diagnostics now fail?

In [ ]:
# Task 1: trace plot, one colour per chain
# ggplot(draws, aes(x = .iteration, y = p, colour = factor(.chain))) +
#   geom_line(alpha = 0.7) + labs(colour = "chain")

# Task 2: the numbers
fit$summary("p", "mean", "sd", "mcse_mean", "rhat", "ess_bulk", "ess_tail")
fit$diagnostic_summary()

| diagnostic | your value | OK? |
|---|---|---|
| R-hat | | |
| bulk ESS / tail ESS | | |
| MCSE of the mean | | |
| divergences | | |
| max treedepth hits | | |
| E-BFMI | | |

**Your answer:** write a few sentences here.

## B3 (15 min): change the prior

**The setting.** As on Day 2 (Exercise A2), a second person starts from the prior **Beta(5, 5)**: mean 0.5, worth about 10 earlier trials. The exact posterior is then Beta(5 + 7, 5 + 3) = **Beta(12, 8)**.

In Stan the prior settings are *data*, so changing the prior only changes the data list: **no new compilation is needed**.

**Tasks.**

1. Run the model again with a = 5, b = 5. Check the diagnostics quickly (R-hat, ESS, divergences).
2. Compare the posterior mean and 90% interval of p with the exact Beta(12, 8): mean 12 / 20, interval `qbeta(c(0.05, 0.95), 12, 8)`.
3. The generated quantity `y_new` is the number of successes in 10 new trials. Estimate P(y_new ≥ 8) under both priors: `mean(draws$y_new >= 8)`.
4. Is the change between the two priors a computation problem or a modelling choice? What would you write in a report about it?

In [ ]:
fit_alt <- mod$sample(
  data   = list(n = 10, y = 7, a = 5, b = 5),    # only the prior changed
  chains = 4, seed = 2026, refresh = 0
)
draws_alt <- fit_alt$draws(format = "df")

# Task 1: fit_alt$summary("p");  fit_alt$diagnostic_summary()
# Task 2: mean(draws_alt$p);  quantile(draws_alt$p, c(0.05, 0.95))
# Task 3: mean(draws$y_new >= 8);  mean(draws_alt$y_new >= 8)

**Your answer:** write a few sentences here.

## B4 (25 min): challenge, no template — your own Stan program for two rates

*Slides "A Stan program has blocks", "Running it from R", "Six numbers to check after every fit" and "R-hat: do the chains agree?".* Write a Stan program from scratch. Plan the blocks in your group first.

**The problem.** Emails per hour in eight one-hour periods, in two inboxes (made-up numbers): personal **3, 5, 2, 4, 6, 3, 4, 5**; work **9, 12, 7, 10, 14, 8, 11, 9**. Model: the counts in inbox k are Poisson with rate λₖ, and each λₖ ~ Gamma(2, 1). Exact posteriors (Day 2): Gamma(34, 9) for the personal and Gamma(82, 9) for the work inbox.

**Tasks.**

1. Write a Stan program for **both inboxes in one model**, with a vector of two rates. In `generated quantities`, add the ratio λ_work / λ_personal and the number of work emails in the next 8 working hours. Save it in `models/`, fit it (4 chains, `seed = 2026`) and check the six numbers.
2. Compare the posterior means and 90% intervals of both rates with the exact Gamma posteriors.
3. Report the 90% interval of the ratio and P(ratio > 2 | y). Then report the 90% interval of the number of work emails in the next 8 hours, and explain why it is wider than the interval of 8 λ_work.

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Blocks:** `models/beta-binomial.stan` shows the four blocks. Data: `int<lower=1> N; array[N] int<lower=0> y; array[N] int<lower=1, upper=2> inbox;`. Parameters: `vector<lower=0>[2] lambda;`.
- **Model block:** `lambda ~ gamma(2, 1);` (Stan's gamma has shape and rate, like R's `dgamma(x, 2, 1)`) and `y ~ poisson(lambda[inbox]);`: each count picks its inbox's rate.
- **Generated quantities:** `real ratio = lambda[2] / lambda[1];` and `int work_next_day = poisson_rng(8 * lambda[2]);`.
- **Data list:** stack the two inboxes: `y = c(personal, work)`, `inbox = rep(1:2, each = 8)`, `N = 16`.
- **Exact intervals:** `qgamma(c(0.05, 0.95), 34, 9)`.
- **Draws:** `fit$draws(format = "df")` gives columns `lambda[1]`, `lambda[2]`, `ratio` and `work_next_day`.

</details>

In [ ]:
# B4: your code

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: Give one example of a computation check and one of a model check from today. What changed when the prior changed, and what did not?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 15:00 for Peer review B.** Save this notebook (Ctrl/Cmd + S). At 15:45: daily diary on Mentimeter.